# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print('Available record sets:')
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '')}")

# List the fields for each record set
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']} ({rs.get('name', '')}) Fields:")
    for field in rs.get('fields', []):
        print(f"   - @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, select the first available record set for demonstration
if len(record_sets) == 0:
    raise ValueError("No record sets are defined in this dataset.")
record_set_id = record_sets[0]['@id']

# If multiple record sets are present, add their ids here
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Columns in DataFrame for record set '{record_set_id}':")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify numeric fields (float/integer) programmatically for the selected record set
df = dataframes[record_set_id]
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if len(numeric_fields) == 0:
    print("No numeric fields found in this record set. Cannot proceed with numeric EDA.")
else:
    numeric_field = numeric_fields[0]  # Choose the first numeric column found
    print(f"Using numeric field for analysis: '{numeric_field}'")

    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical column (if present)
    # Try to find a non-numeric field
    group_candidates = [c for c in df.columns if c not in numeric_fields]
    group_field = group_candidates[0] if group_candidates else None

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_fields) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field is available and categorical
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the [FAIR² Croissant-format mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant`, explored available record sets and fields, and performed basic numerical analysis and visualization. This workflow can be extended for advanced bioinformatics, protein analysis, or application-specific exploratory data analysis.